<a href="https://colab.research.google.com/github/mhowlin-web/TP_RAG_ARCA/blob/main/02_limpieza_encoding_corpus_arca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP RAG y Agentes

## Asistente para consultas sobre trámites de Monotributo en ARCA

En este notebook cargo el corpus obtenido en el Notebook 01 y realizo una
limpieza del texto antes de utilizarlo en el sistema RAG.

El objetivo principal de esta etapa es corregir problemas de codificación
de caracteres que aparecen en algunos textos extraídos de las páginas web.

Por ejemplo, algunos textos pueden aparecer como:

    FacturaciÃ³n apÃ³crifa

cuando deberían aparecer como:

    Facturación apócrifa

Para solucionar este problema utilizo `ftfy`, una librería especializada
en la reparación de texto con problemas de codificación.

Además realizo una limpieza básica del texto, pero sin modificar
innecesariamente el contenido de los documentos.

El resultado será un nuevo archivo JSON que utilizaré como entrada para
las siguientes etapas del RAG.

## Montaje de Google Drive

En esta celda monto Google Drive para trabajar con los mismos archivos que
utilicé en el Notebook 01.

De esta manera puedo mantener el corpus entre diferentes notebooks de
Google Colab sin depender del almacenamiento temporal de la sesión.

In [17]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Definición de la carpeta del proyecto

En esta celda defino la carpeta de Google Drive donde estoy guardando
los notebooks y los archivos del proyecto.

Utilizo una única variable para la carpeta del proyecto para evitar
problemas de rutas entre notebooks.

In [18]:
from pathlib import Path

CARPETA_PROYECTO = Path(
    "/content/drive/MyDrive/TP_RAG_ARCA"
)

CARPETA_PROYECTO.mkdir(
    parents=True,
    exist_ok=True
)

print("Carpeta del proyecto:")
print(CARPETA_PROYECTO)

Carpeta del proyecto:
/content/drive/MyDrive/TP_RAG_ARCA


## Instalación de librerías

En esta celda instalo `ftfy`, que utilizaré para corregir automáticamente
problemas de codificación como `Ã³`, `Ã¡`, `Ã©`, etc.

También utilizo las librerías estándar `json` y `re` para trabajar con
el corpus y realizar la limpieza básica del texto.

In [19]:
!pip install -q ftfy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00


## Importación de librerías

En esta celda importo las librerías que utilizaré durante la limpieza
del corpus.

In [20]:
import json
import re

from pathlib import Path
from ftfy import fix_text

## Definición de los archivos

En esta celda defino el archivo JSON generado en el Notebook 01 y el
archivo JSON que voy a generar después de corregir el texto.

Mantengo el archivo original sin modificar para poder volver a él si
fuera necesario.

El nuevo archivo será utilizado en las siguientes etapas del proyecto RAG.

In [21]:
ARCHIVO_ORIGINAL = (
    CARPETA_PROYECTO /
    "corpus" /
    "corpus_arca_monotributo.json"
)

ARCHIVO_LIMPIO = (
    CARPETA_PROYECTO /
    "corpus" /
    "corpus_arca_monotributo_limpio.json"
)

print("Archivo original:")
print(ARCHIVO_ORIGINAL)

print("\nArchivo limpio:")
print(ARCHIVO_LIMPIO)

Archivo original:
/content/drive/MyDrive/TP_RAG_ARCA/corpus/corpus_arca_monotributo.json

Archivo limpio:
/content/drive/MyDrive/TP_RAG_ARCA/corpus/corpus_arca_monotributo_limpio.json


## Verificación de la existencia del corpus

En esta celda verifico que el archivo generado en el Notebook 01 exista
antes de intentar cargarlo.

Si esta celda produce un error, significa que la ruta definida anteriormente
no coincide con la ubicación real del archivo en Google Drive.

In [22]:
if not ARCHIVO_ORIGINAL.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo:\n{ARCHIVO_ORIGINAL}"
    )

print("Archivo encontrado correctamente.")
print(f"Tamaño: {ARCHIVO_ORIGINAL.stat().st_size:,} bytes")

Archivo encontrado correctamente.
Tamaño: 28,816 bytes


## Carga del corpus original

En esta celda cargo el JSON generado en el Notebook 01.

Utilizo `encoding="utf-8"` porque el archivo JSON está almacenado utilizando
codificación UTF-8.

In [23]:
with open(
    ARCHIVO_ORIGINAL,
    "r",
    encoding="utf-8"
) as archivo:

    corpus_original = json.load(archivo)

print(
    f"Documentos cargados: {len(corpus_original)}"
)

Documentos cargados: 8


## Inspección del texto original

En esta celda muestro una parte del corpus original para detectar
problemas de codificación antes de realizar la corrección.

Esto me permite comparar posteriormente el texto original con el texto
corregido.

In [24]:
for documento in corpus_original:

    print("=" * 80)
    print(documento["titulo"])
    print("=" * 80)

    print(documento["texto"][:2000])

    print("\n")

Inicio - Ayuda sobre el Monotributo
Inicio - Ayuda sobre el monotributo - Monotributo | ARCA
Evitar las herramientas de navegaciÃ³n y pasar al contenido
Monotributo
Menu
Inicio
Ayuda
Inicio
Ayuda sobre el monotributo
Inicio
Ayuda sobre el monotributo
Toda la informaciÃ³n sobre cÃ³mo darte de alta y hacer operaciones como monotributista.
Ingresar con clave fiscal
MenÃº de contenidos
QuÃ© es
INSCRIPCIÃN
Inicio
Clave fiscal
CUIT
Domicilio Fiscal ElectrÃ³nico
Jurisdicciones
Actividades
ALTA DE MONOTRIBUTO
Procedimiento
Tipos de monotributo
ParÃ¡metros
JubilaciÃ³n
Obra social
Monotributo unificado
Constancias y credenciales
DESPUÃS DEL ALTA
Desarrollo de la actividad
FacturaciÃ³n
Pagos
RecategorizaciÃ³n
FINALIZACIÃN DE ACTIVIDADES
Baja
Por cese de actividades
De oficio
ExclusiÃ³n
Renuncia
Pasaje al rÃ©gimen general
Ayuda
Inicio
El primer paso es inscribirse ante ARCA para poder despuÃ©s darse de alta en impuestos y utilizar los servicios con clave fiscal.
Para ello es necesario obtener l

## Corrección automática de problemas de codificación

En esta celda creo una función que utiliza `ftfy` para reparar problemas
de codificación de caracteres.

No intento reemplazar manualmente caracteres como `Ã³` por `ó`.

La librería analiza el texto y realiza la corrección automáticamente,
lo que permite aplicar el mismo procedimiento a todo el corpus.

In [25]:
def corregir_encoding(texto):
    """
    Corrige problemas comunes de codificación Unicode
    utilizando ftfy.
    """

    if not isinstance(texto, str):
        return texto

    return fix_text(texto)

## Limpieza básica del texto

En esta celda creo una segunda función para realizar una limpieza
conservadora del texto.

El objetivo no es resumir ni reescribir los documentos, sino eliminar
ruido de formato como espacios repetidos y líneas vacías excesivas.

Mantengo el contenido semántico porque posteriormente será utilizado
para generar embeddings.

In [26]:
def limpiar_texto(texto):

    if not isinstance(texto, str):
        return texto

    # Primero corrijo problemas de encoding
    texto = corregir_encoding(texto)

    # Normalizo espacios horizontales
    texto = re.sub(
        r"[ \t]+",
        " ",
        texto
    )

    # Reduzco líneas vacías consecutivas
    texto = re.sub(
        r"\n\s*\n+",
        "\n\n",
        texto
    )

    return texto.strip()

## Aplicación de la limpieza al corpus

En esta celda aplico la corrección de encoding y la limpieza básica a
todos los documentos del corpus.

No modifico los metadatos originales del documento.

Solamente reemplazo el contenido de `texto` por su versión corregida.

In [27]:
corpus_limpio = []

for documento in corpus_original:

    documento_limpio = documento.copy()

    documento_limpio["texto"] = limpiar_texto(
        documento["texto"]
    )

    corpus_limpio.append(
        documento_limpio
    )

print(
    f"Documentos procesados: {len(corpus_limpio)}"
)

Documentos procesados: 8


## Comparación antes y después

En esta celda comparo una parte del mismo documento antes y después
de aplicar la corrección.

Esto me permite verificar visualmente si problemas como `FacturaciÃ³n`
fueron solucionados.

In [28]:
documento_original = corpus_original[0]
documento_limpio = corpus_limpio[0]

print("=" * 80)
print("TEXTO ORIGINAL")
print("=" * 80)

print(
    documento_original["texto"][:5000]
)

print("\n")
print("=" * 80)
print("TEXTO CORREGIDO")
print("=" * 80)

print(
    documento_limpio["texto"][:5000]
)

TEXTO ORIGINAL
Inicio - Ayuda sobre el monotributo - Monotributo | ARCA
Evitar las herramientas de navegaciÃ³n y pasar al contenido
Monotributo
Menu
Inicio
Ayuda
Inicio
Ayuda sobre el monotributo
Inicio
Ayuda sobre el monotributo
Toda la informaciÃ³n sobre cÃ³mo darte de alta y hacer operaciones como monotributista.
Ingresar con clave fiscal
MenÃº de contenidos
QuÃ© es
INSCRIPCIÃN
Inicio
Clave fiscal
CUIT
Domicilio Fiscal ElectrÃ³nico
Jurisdicciones
Actividades
ALTA DE MONOTRIBUTO
Procedimiento
Tipos de monotributo
ParÃ¡metros
JubilaciÃ³n
Obra social
Monotributo unificado
Constancias y credenciales
DESPUÃS DEL ALTA
Desarrollo de la actividad
FacturaciÃ³n
Pagos
RecategorizaciÃ³n
FINALIZACIÃN DE ACTIVIDADES
Baja
Por cese de actividades
De oficio
ExclusiÃ³n
Renuncia
Pasaje al rÃ©gimen general
Ayuda
Inicio
El primer paso es inscribirse ante ARCA para poder despuÃ©s darse de alta en impuestos y utilizar los servicios con clave fiscal.
Para ello es necesario obtener la
clave fiscal
y la
C

## Búsqueda de caracteres sospechosos

En esta celda busco algunas secuencias que suelen aparecer cuando
un texto UTF-8 fue interpretado incorrectamente como otra codificación.

No utilizo esta búsqueda para corregir el texto.

La utilizo solamente como control de calidad para comprobar si todavía
quedan problemas evidentes después de utilizar `ftfy`.

In [29]:
PATRONES_SOSPECHOSOS = [
    "Ã",
    "Â",
    "â€",
    "ðŸ",
    "�"
]

for documento in corpus_limpio:

    texto = documento["texto"]

    encontrados = [
        patron
        for patron in PATRONES_SOSPECHOSOS
        if patron in texto
    ]

    if encontrados:

        print(
            f"\nDocumento: {documento['titulo']}"
        )

        print(
            "Patrones encontrados:",
            encontrados
        )

## Revisión de palabras que tenían problemas de encoding

En esta celda puedo buscar específicamente ejemplos que aparecían
incorrectamente en la extracción original.

Por ejemplo, busco la palabra `Facturación`.

Si la corrección funcionó, debería encontrar la palabra correctamente
codificada en el corpus limpio.

In [30]:
terminos_prueba = [
    "Facturación",
    "apócrifa",
    "Constatación"
]

for termino in terminos_prueba:

    encontrados = []

    for documento in corpus_limpio:

        if termino.lower() in documento["texto"].lower():

            encontrados.append(
                documento["titulo"]
            )

    print(
        f"{termino}: {len(encontrados)} documentos"
    )

    for documento in encontrados:
        print(f"  - {documento}")

Facturación: 8 documentos
  - Inicio - Ayuda sobre el Monotributo
  - Obtención de Clave Fiscal
  - Constancias y credenciales
  - Facturación
  - Recategorización
  - Baja de monotributo
  - Desarrollo de la actividad
  - Tutoriales sobre Monotributo
apócrifa: 8 documentos
  - Inicio - Ayuda sobre el Monotributo
  - Obtención de Clave Fiscal
  - Constancias y credenciales
  - Facturación
  - Recategorización
  - Baja de monotributo
  - Desarrollo de la actividad
  - Tutoriales sobre Monotributo
Constatación: 8 documentos
  - Inicio - Ayuda sobre el Monotributo
  - Obtención de Clave Fiscal
  - Constancias y credenciales
  - Facturación
  - Recategorización
  - Baja de monotributo
  - Desarrollo de la actividad
  - Tutoriales sobre Monotributo


## Guardado del corpus limpio

En esta celda guardo el corpus corregido en un nuevo archivo JSON.

Mantengo el archivo original separado del archivo limpio para conservar
una copia del corpus tal como fue descargado originalmente.

El archivo limpio será la entrada para la siguiente etapa del proyecto.

In [31]:
with open(
    ARCHIVO_LIMPIO,
    "w",
    encoding="utf-8"
) as archivo:

    json.dump(
        corpus_limpio,
        archivo,
        ensure_ascii=False,
        indent=2
    )

print(
    "Corpus limpio guardado correctamente:"
)

print(
    ARCHIVO_LIMPIO
)

Corpus limpio guardado correctamente:
/content/drive/MyDrive/TP_RAG_ARCA/corpus/corpus_arca_monotributo_limpio.json


## Verificación del archivo generado

En esta celda vuelvo a cargar el archivo limpio para comprobar que
el JSON puede ser leído correctamente después de haber sido guardado.

También verifico que conserve la misma cantidad de documentos que
el corpus original.

In [32]:
with open(
    ARCHIVO_LIMPIO,
    "r",
    encoding="utf-8"
) as archivo:

    corpus_verificado = json.load(
        archivo
    )

print(
    "Documentos originales:",
    len(corpus_original)
)

print(
    "Documentos en el archivo limpio:",
    len(corpus_verificado)
)

if len(corpus_original) == len(corpus_verificado):

    print("\nVerificación OK.")

else:

    raise ValueError(
        "La cantidad de documentos no coincide."
    )

Documentos originales: 8
Documentos en el archivo limpio: 8

Verificación OK.


## Inspección final

En esta celda muestro nuevamente una parte del corpus limpio.

Después de esta revisión, considero que el corpus está preparado para
la siguiente etapa del proyecto RAG.

En el próximo notebook voy a trabajar con los documentos limpios para
dividirlos en chunks y preparar los fragmentos que posteriormente
utilizaré para generar embeddings y almacenarlos en Pinecone.

In [33]:
documento = corpus_verificado[0]

print("=" * 80)
print(documento["titulo"])
print("=" * 80)

print(
    documento["texto"][:10000]
)

Inicio - Ayuda sobre el Monotributo
Inicio - Ayuda sobre el monotributo - Monotributo | ARCA
Evitar las herramientas de navegación y pasar al contenido
Monotributo
Menu
Inicio
Ayuda
Inicio
Ayuda sobre el monotributo
Inicio
Ayuda sobre el monotributo
Toda la información sobre cómo darte de alta y hacer operaciones como monotributista.
Ingresar con clave fiscal
Menú de contenidos
Qué es
INSCRIPCIÓN
Inicio
Clave fiscal
CUIT
Domicilio Fiscal Electrónico
Jurisdicciones
Actividades
ALTA DE MONOTRIBUTO
Procedimiento
Tipos de monotributo
Parámetros
Jubilación
Obra social
Monotributo unificado
Constancias y credenciales
DESPUÉS DEL ALTA
Desarrollo de la actividad
Facturación
Pagos
Recategorización
FINALIZACIÓN DE ACTIVIDADES
Baja
Por cese de actividades
De oficio
Exclusión
Renuncia
Pasaje al régimen general
Ayuda
Inicio
El primer paso es inscribirse ante ARCA para poder después darse de alta en impuestos y utilizar los servicios con clave fiscal.
Para ello es necesario obtener la
clave fiscal
y

## Conclusión

En este notebook cargué el corpus obtenido desde las fuentes oficiales
de ARCA y corregí automáticamente problemas de codificación de caracteres.

Utilicé `ftfy` para reparar textos que podían aparecer como:

    FacturaciÃ³n apÃ³crifa

en lugar de:

    Facturación apócrifa

También realicé una limpieza básica del formato sin modificar
deliberadamente el contenido semántico de los documentos.

El corpus original se conserva sin modificaciones y el resultado
limpio se guarda como:

    corpus_arca_monotributo_limpio.json

Este archivo será utilizado como entrada para la siguiente etapa
del proyecto RAG.